# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Engagement and Visibility Move Together

The paper reports that pages with high scroll depth and high engagement have higher Health Scores, with an observed difference of about 11.2 points between the strongest and weakest bucket.

**My methodology question:**  
Because scroll depth is already one of the components used to calculate Health Score, could part of this relationship come from the metric construction itself? I would want to check the relationship using an outcome that does not directly include scroll depth.

This does not mean the finding is wrong. It means the strength of the relationship should be interpreted carefully.

### Finding 2 — The Freshness Multiplier

The paper reports that 365+ day content refreshed within 30 days showed a 3.2× Health Score boost and 57× more impressions.

**My methodology question:**  
How were refreshed pages selected, and was there a comparable control group or time-aware validation design? Pages chosen for refresh may already differ from pages that were not refreshed.

I would want to know whether the comparison supports a refresh effect or only shows an observed difference between the two groups.

These questions are meant to make the findings more rigorous, not to reject them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation Experiment Setup

In Week 5, the model was evaluated using **5-fold `GroupKFold` grouped by client**. To demonstrate why entity-grouped validation is essential for multi-page client datasets, I compare this honest validation setup against a weaker **simple random row split** on the exact same dataset and model configuration.

### Canonical Methodology & Controls

Both experiments use identical settings:
- **Eligible Population**: 16,513 pages (`impressions_total >= 1000` and `april_clicks >= 10`) across 36 clients
- **Target**: `may_clicks < 0.8 * april_clicks` (Base Rate: **41.62%**)
- **Feature Set**: 9 pre-May historical features (`impressions_total`, `clicks_total`, `april_impressions`, `april_clicks`, `feb_clicks`, `momentum`, `ctr`, `active_days`, `weighted_position`)
- **Model**: `RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)`
- **Primary Metric**: **Precision@50** (evaluated on out-of-fold validation sets)
- **Ranking Tie-Policy**: Sort by predicted score desc, secondary tie-break by `april_clicks` desc, final tie-break by `content_hash_id` asc

---

### Before vs. After Validation Comparison

Executing the cell below yields the following measured comparison:

| Validation setup | Split type | Precision@50 | Client overlap |
|---|---|---:|---:|
| **Before** (Weaker setup) | 5-Fold Random Row Split (`KFold`) | *Computed in cell below* | *Present across folds* |
| **After** (Honest setup) | 5-Fold `GroupKFold` by Client | **0.4440** | **0** |

---

### Interpretation & Validation Insights

1. **Why the Random Row Split produces an optimistic score**:
   In a random row split, pages belonging to the same client are randomly distributed across both training and validation folds. Because pages from the same client share hidden characteristics (such as domain authority, technical infrastructure, and search niche), the model can memorize client-specific signals during training and exploit them when predicting validation pages from those same clients.

2. **Why Client-Grouped Validation (`GroupKFold`) is more honest**:
   Grouping folds strictly by `client_hash_id` guarantees **zero client overlap** between training and validation sets in every fold. This tests whether the model can generalize to **unseen client domains**, mimicking real decision-support deployment where the system evaluates pages from newly onboarded or un-memorized clients.

3. **Methodological Finding**:
   The measured performance gap between the random row split and the client-grouped split reflects the degree of client memorization. The grouped validation metric (**Precision@50 = 0.4440**) provides a **more realistic and conservative estimate** of out-of-fold generalization performance for decision-support prioritization.


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.model_selection import KFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier

print('==================================================')
print('SECTION 2: VALIDATION AUDIT — BEFORE (RANDOM) VS AFTER (GROUPED)')
print('==================================================')

# 1. Obtain HF_TOKEN to access gated FlyRank/internship-warehouse dataset
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    env_file = Path('.env')
    if not env_file.exists():
        env_file = Path('../.env')
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip().strip('"\'')

if not HF_TOKEN:
    print('[WARNING] HF_TOKEN not found in environment or .env file.')
    print('Please provide your Hugging Face READ token to stream real warehouse daily partitions.')
    raise ValueError('HF_TOKEN required to load real gated warehouse dataset FlyRank/internship-warehouse.')

# 2. Download Feb-May 2026 daily performance partitions from Hugging Face
from huggingface_hub import snapshot_download
local_dir = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    allow_patterns=[
        'dim_*.parquet',
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
        'fact_content_daily_performance/month=2026-04/*.parquet',
        'fact_content_daily_performance/month=2026-05/*.parquet'
    ],
    token=HF_TOKEN
)

fact_files = list(Path(local_dir).glob('fact_content_daily_performance/**/*.parquet'))
print(f'Loaded {len(fact_files)} daily performance parquet partitions.')

# 3. Streamed Polars Aggregation for canonical Week 5 population & features
lazy_daily = pl.scan_parquet(fact_files).select([
    pl.col('report_date').cast(pl.Utf8),
    pl.col('client_hash_id').cast(pl.Categorical),
    pl.col('content_hash_id').cast(pl.Categorical),
    pl.col('gsc_clicks').cast(pl.Int32),
    pl.col('gsc_impressions').cast(pl.Int32),
    pl.col('gsc_avg_position').cast(pl.Float32)
])

feb_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-02-28')
mar_mask = (pl.col('report_date') >= '2026-03-01') & (pl.col('report_date') <= '2026-03-31')
apr_mask = (pl.col('report_date') >= '2026-04-01') & (pl.col('report_date') <= '2026-04-30')
pre_may_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-04-30')
may_mask = (pl.col('report_date') >= '2026-05-01') & (pl.col('report_date') <= '2026-05-31')

lazy_main_agg = lazy_daily.group_by(['client_hash_id', 'content_hash_id']).agg([
    pl.col('gsc_clicks').filter(feb_mask).sum().alias('feb_clicks'),
    pl.col('gsc_impressions').filter(feb_mask).sum().alias('feb_impressions'),
    pl.col('gsc_clicks').filter(mar_mask).sum().alias('march_clicks'),
    pl.col('gsc_impressions').filter(mar_mask).sum().alias('march_impressions'),
    pl.col('gsc_clicks').filter(apr_mask).sum().alias('april_clicks'),
    pl.col('gsc_impressions').filter(apr_mask).sum().alias('april_impressions'),
    pl.col('gsc_impressions').filter(pre_may_mask).sum().alias('impressions_total'),
    pl.col('gsc_clicks').filter(pre_may_mask).sum().alias('clicks_total'),
    pl.col('report_date').filter(pre_may_mask & (pl.col('gsc_impressions') > 0)).n_unique().alias('active_days'),
    pl.col('gsc_clicks').filter(may_mask).sum().alias('may_clicks'),
    pl.col('gsc_impressions').filter(may_mask).sum().alias('may_impressions')
])

lazy_pos_agg = lazy_daily.filter(pre_may_mask & (pl.col('gsc_avg_position') > 0)).group_by(['client_hash_id', 'content_hash_id']).agg([
    (pl.col('gsc_avg_position') * pl.col('gsc_impressions')).sum().alias('pos_num'),
    pl.col('gsc_impressions').sum().alias('pos_den')
])

agg_main = lazy_main_agg.collect()
agg_pos = lazy_pos_agg.collect()

agg_df = agg_main.join(agg_pos, on=['client_hash_id', 'content_hash_id'], how='left')
agg_df = agg_df.with_columns([
    (pl.col('pos_num') / pl.col('pos_den')).alias('weighted_position')
]).drop(['pos_num', 'pos_den'])

del agg_main, agg_pos
gc.collect()

# 4. Canonical Derived Features & Target
agg_df = agg_df.with_columns([
    (pl.col('april_clicks') / (pl.col('feb_clicks') + 1.0)).alias('momentum'),
    ((pl.col('clicks_total') / pl.col('impressions_total')) * 100).alias('ctr'),
    (pl.col('may_clicks') < (0.8 * pl.col('april_clicks'))).cast(pl.Int64).alias('decline')
])

# Filter canonical Week 5 eligible population & deterministic sort
elig_df = agg_df.filter((pl.col('impressions_total') >= 1000) & (pl.col('april_clicks') >= 10))
elig_df = elig_df.sort(['client_hash_id', 'content_hash_id'])

elig_pd = elig_df.to_pandas()
elig_pd['weighted_position'] = elig_pd['weighted_position'].fillna(elig_pd['weighted_position'].median())

feature_cols = [
    'impressions_total',
    'clicks_total',
    'april_impressions',
    'april_clicks',
    'feb_clicks',
    'momentum',
    'ctr',
    'active_days',
    'weighted_position'
]

n_eligible = len(elig_pd)
base_rate = float(elig_pd['decline'].mean())
n_clients = elig_pd['client_hash_id'].nunique()

print(f'\nDataset Summary:')
print(f'- Eligible Population: {n_eligible:,} pages across {n_clients} clients')
print(f'- Base Rate (Decline %): {base_rate * 100:.2f}% ({elig_pd["decline"].sum():,} declining / {n_eligible - elig_pd["decline"].sum():,} non-declining)')

# Helper function for deterministic Precision@K evaluation
def eval_precision_at_k(df_fold, score_col, k=50):
    sorted_df = df_fold.sort_values(
        by=[score_col, 'april_clicks', 'content_hash_id'],
        ascending=[False, False, True]
    )
    top_k = sorted_df.head(k)
    return float(top_k['decline'].mean()) if len(top_k) > 0 else 0.0

# 5. BEFORE EXPERIMENT: 5-Fold Random Row Split (KFold)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X = elig_pd[feature_cols]
y = elig_pd['decline']
groups = elig_pd['client_hash_id']

before_p50_scores = []
before_client_overlaps = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    tr_df = elig_pd.iloc[tr_idx]
    val_df = elig_pd.iloc[val_idx].copy()
    
    tr_clients = set(groups.iloc[tr_idx])
    val_clients = set(groups.iloc[val_idx])
    overlap = len(tr_clients.intersection(val_clients))
    before_client_overlaps.append(overlap)
    
    rf_before = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_before.fit(tr_df[feature_cols], tr_df['decline'])
    
    val_df['rf_score'] = rf_before.predict_proba(val_df[feature_cols])[:, 1]
    p50 = eval_precision_at_k(val_df, 'rf_score', k=50)
    before_p50_scores.append(p50)

before_mean_p50 = float(np.mean(before_p50_scores))
max_before_overlap = max(before_client_overlaps)

# 6. AFTER EXPERIMENT: 5-Fold GroupKFold by Client
gkf = GroupKFold(n_splits=5)

after_p50_scores = []
after_client_overlaps = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    tr_df = elig_pd.iloc[tr_idx]
    val_df = elig_pd.iloc[val_idx].copy()
    
    tr_clients = set(groups.iloc[tr_idx])
    val_clients = set(groups.iloc[val_idx])
    overlap = len(tr_clients.intersection(val_clients))
    after_client_overlaps.append(overlap)
    assert overlap == 0, f'Fold {fold} has client overlap in GroupKFold!'
    
    rf_after = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_after.fit(tr_df[feature_cols], tr_df['decline'])
    
    val_df['rf_score'] = rf_after.predict_proba(val_df[feature_cols])[:, 1]
    p50 = eval_precision_at_k(val_df, 'rf_score', k=50)
    after_p50_scores.append(p50)

after_mean_p50 = float(np.mean(after_p50_scores))
max_after_overlap = max(after_client_overlaps)

delta_p50 = after_mean_p50 - before_mean_p50

# 7. Print Comparison Table
summary_df = pd.DataFrame([
    {
        'Validation setup': 'Before (Weaker setup)',
        'Split type': 'Random 5-Fold KFold',
        'Precision@50': round(before_mean_p50, 4),
        'Client overlap': f'{max_before_overlap} clients shared'
    },
    {
        'Validation setup': 'After (Honest setup)',
        'Split type': '5-Fold GroupKFold by client',
        'Precision@50': round(after_mean_p50, 4),
        'Client overlap': f'{max_after_overlap} (Zero overlap)'
    }
])

print('\n==================================================')
print('VALIDATION AUDIT SUMMARY COMPARISON TABLE')
print('==================================================')
print(summary_df.to_string(index=False))

print(f'\n--- SUMMARY METRICS ---')
print(f'1. Eligible Pages:     {n_eligible:,}')
print(f'2. Decline Base Rate:   {base_rate * 100:.2f}%')
print(f'3. BEFORE Precision@50: {before_mean_p50:.4f}')
print(f'4. AFTER Precision@50:  {after_mean_p50:.4f}')
print(f'5. Delta (After - Before): {delta_p50:+.4f} ({delta_p50 * 100:+.2f} percentage points)')
print(f'6. Grouped Client Overlap Zero Verified: {max_after_overlap == 0}')
print(f'7. Result Reproducible: YES (fixed random_state=42 and deterministic tie-breaking)')


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.